In [27]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output
import time
import networkx as nx
import matplotlib.pyplot as plt
from rl4co.envs import FJSPEnv
from rl4co.models.zoo.l2d import L2DModel
from rl4co.models.zoo.l2d.policy import L2DPolicy
from rl4co.models.zoo.l2d.decoder import L2DDecoder
from rl4co.models.nn.graph.hgnn import HetGNNEncoder
from rl4co.utils.trainer import RL4COTrainer


import utils
import diffusion
import inference
import importlib
importlib.reload(diffusion)
importlib.reload(inference)
importlib.reload(utils)


"""
is_ready
start_op_per_job
num_eligible
next_op
ma_assignment
lbs

td :
TensorDict(
    fields={
        action_mask: Tensor(shape=torch.Size([1, 26]), device=cpu, dtype=torch.bool, is_shared=False),
        busy_until: Tensor(shape=torch.Size([1, 5]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([1, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        end_op_per_job: Tensor(shape=torch.Size([1, 5]), device=cpu, dtype=torch.int64, is_shared=False),
        finish_times: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        is_ready: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.bool, is_shared=False),
        job_done: Tensor(shape=torch.Size([1, 5]), device=cpu, dtype=torch.bool, is_shared=False),
        job_in_process: Tensor(shape=torch.Size([1, 5]), device=cpu, dtype=torch.bool, is_shared=False),
        job_ops_adj: Tensor(shape=torch.Size([1, 5, 10]), device=cpu, dtype=torch.int64, is_shared=False),
        lbs: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        ma_assignment: Tensor(shape=torch.Size([1, 5, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        next_op: Tensor(shape=torch.Size([1, 5]), device=cpu, dtype=torch.int64, is_shared=False),
        num_eligible: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        op_scheduled: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.bool, is_shared=False),
        ops_adj: Tensor(shape=torch.Size([1, 10, 10, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        ops_job_map: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.int64, is_shared=False),
        ops_ma_adj: Tensor(shape=torch.Size([1, 5, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        ops_sequence_order: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.int64, is_shared=False),
        pad_mask: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.bool, is_shared=False),
        proc_times: Tensor(shape=torch.Size([1, 5, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        reward: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
        start_op_per_job: Tensor(shape=torch.Size([1, 5]), device=cpu, dtype=torch.int64, is_shared=False),
        start_times: Tensor(shape=torch.Size([1, 10]), device=cpu, dtype=torch.float32, is_shared=False),
        terminated: Tensor(shape=torch.Size([1, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        time: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([1]),
    device=None,
    is_shared=False)

"""




generator_params = {
"num_jobs": 4,  # the total number of jobs
"num_machines": 4,  # the total number of machines that can process operations
"min_ops_per_job": 4,  # minimum number of operatios per job
"max_ops_per_job": 4,  # maximum number of operations per job
"min_processing_time": 5,  # the minimum time required for a machine to process an operation
"max_processing_time": 20,  # the maximum time required for a machine to process an operation
"min_eligible_ma_per_op": 1,  # the minimum number of machines capable to process an operation
"max_eligible_ma_per_op": 2,  # the maximum number of machines capable to process an operation
}

batch_size = 32
embed_dim = 32
encoder_num_layers = 4
epochs = 100

# 1, 20, 16

# Encoding operations and machines
env = FJSPEnv(generator_params=generator_params)
td = env.reset(batch_size=[batch_size])
encoder = HetGNNEncoder(embed_dim=embed_dim, num_layers=encoder_num_layers)
(ma_emb, op_emb), init = encoder(td)
print("Shapes of original embeddings")
print(f"Ma: {ma_emb.shape}")
print(f"Op: {op_emb.shape}")

# Combining embeddings
dataset = utils.EmbeddingDataset(op_emb, ma_emb, td, env)
print(f"Shape of a combined embedding instance: {dataset[0].shape}")

# Diffusion process
model_path = 'trained_diffusion_model.pth'
# diffusion.apply_diffusion(batch_size, dataset, epochs, model_path)

def make_samples():
    # Applying inference
    dim_x = op_emb.shape[1] + ma_emb.shape[1]
    dim_y = embed_dim
    n_samples = 32
    samples = inference.apply_inference(dim_x, dim_y, model_path, n_samples)
    print(f"Shapes of an inferenced sample instance: {samples.shape}")
    return samples


def makespan_of_generated_schedules(samples):

    # Decompositioning op and ma embeddings
    s_op_emb, s_ma_emb = dataset.decomposition_instance(samples)
    print(f"Decompositioned op embedding: {s_op_emb.shape}")
    print(f"Decompositioned ma embedding: {s_ma_emb.shape}")
        
    # Visulizing schedule
    makespan_batches = utils.visulize_schedule(td, env, embed_dim, s_ma_emb, s_op_emb)
    return makespan_batches

def avg_makespan_of_bacthes(makespans):
    return makespans.mean()

samples = make_samples()
makespan_batches = makespan_of_generated_schedules(samples)


# 80 x2
# 81
#79
#75
#76
# 78.5

# 75
# 79 x2
#78
#83
##77
# 78.5

Shapes of original embeddings
Ma: torch.Size([32, 16, 32])
Op: torch.Size([32, 4, 32])
Shape of a combined embedding instance: torch.Size([1, 20, 32])
Making samples
torch.Size([32, 1, 20, 32])
Shapes of an inferenced sample instance: torch.Size([32, 1, 20, 32])
Decompositioned op embedding: torch.Size([32, 4, 32])
Decompositioned ma embedding: torch.Size([32, 16, 32])
ma og op emb size
torch.Size([32, 16, 32])
torch.Size([32, 4, 32])


In [28]:
print(makespan_batches)
avg_makespan = avg_makespan_of_bacthes(makespan_batches)
print(avg_makespan)

tensor([ 65.,  87.,  73.,  67.,  75.,  70.,  72.,  67.,  74.,  78.,  92.,  67.,
         71.,  73.,  75.,  85.,  81., 116.,  86., 107.,  75.,  88.,  61.,  62.,
         78.,  81.,  77.,  58.,  81.,  63.,  69.,  77.])
tensor(76.5938)
